# Frozen nominal compiler: structured robustness stress test

This notebook evaluates the trained nominal compiler without updating its weights. It reproduces the one-dimensional perturbation sweeps, two-dimensional RF/B0 maps, and joint routine/conservative uncertainty tests used in the paper.

Run `01_train_nominal.ipynb` first. This notebook reads that nominal-training export directly; set the `NOMINAL_ARCHIVE` environment variable if the archive is stored elsewhere.

In [ ]:
import os, sys, math, json, shutil, zipfile, hashlib, platform, subprocess
from pathlib import Path
from dataclasses import dataclass, fields
from datetime import datetime
import numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib.pyplot as plt

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the full stress test.")
DEVICE=torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32=True
print("Python:",sys.version); print("PyTorch:",torch.__version__); print("GPU:",torch.cuda.get_device_name(0))

def sha256_file(p,block=1024*1024):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(block),b''): h.update(c)
    return h.hexdigest()

REPO_ROOT=Path.cwd().resolve()
if REPO_ROOT.name=='notebooks': REPO_ROOT=REPO_ROOT.parent
explicit=os.environ.get('NOMINAL_ARCHIVE')
candidates=sorted((REPO_ROOT/'outputs').glob('full_haar_nominal_paper_export_*.zip'))
if explicit:
    INPUT_ZIP=Path(explicit).expanduser().resolve()
elif candidates:
    INPUT_ZIP=candidates[-1]
else:
    raise FileNotFoundError('Run 01_train_nominal.ipynb first, or set NOMINAL_ARCHIVE to the nominal export ZIP.')

ROOT=REPO_ROOT/'outputs'/'stress_workspace'
shutil.rmtree(ROOT,ignore_errors=True); ROOT.mkdir(parents=True)
EXTRACTED=ROOT/'nominal_archive'; EXTRACTED.mkdir()
with zipfile.ZipFile(INPUT_ZIP,'r') as zf: zf.extractall(EXTRACTED)

def first_match(patterns):
    for pat in patterns:
        hits=list(EXTRACTED.rglob(pat))
        if hits: return hits[0]
    return None

CONFIG_PATH=first_match(['config.json','nominal_config.json'])
BEST_PATH=first_match(['best_full_checkpoint.pt','best.pt','nominal_frozen_checkpoint.pt'])
if CONFIG_PATH is None or BEST_PATH is None:
    raise FileNotFoundError('The nominal export must contain the model config and selected checkpoint.')
with open(CONFIG_PATH) as f: CONFIG_DICT=json.load(f)
print('Nominal archive:',INPUT_ZIP); print('SHA256:',sha256_file(INPUT_ZIP))

In [ ]:
@dataclass
class Config:
    v_hz: tuple=(-921.0,40.75,700.0); j_hz: tuple=(-64.0,24.4,34.1); n_slices:int=300; dt_s:float=35e-6; rabi_max_hz:float=50000.0
    width:int=256; hidden_layers:int=10; dropout:float=.10; batch_size:int=32; train_steps:int=5000; lr:float=5e-4; weight_decay:float=1e-3; grad_clip:float=1.0
    smooth_lambda:float=1e-4; power_lambda:float=1e-6; val_gates:int=512; val_batch:int=64; validate_every:int=100; print_every:int=20; seed:int=20260917; out_dir:str='full_haar_nominal_run'
valid={f.name for f in fields(Config)}; CFG=Config(**{k:v for k,v in CONFIG_DICT.items() if k in valid}); CFG.v_hz=tuple(CFG.v_hz); CFG.j_hz=tuple(CFG.j_hz); print(CFG)
def kron3(a,b,c): return torch.kron(torch.kron(a,b),c)
class NMRSystem:
    def __init__(self,cfg,device):
        ct=torch.complex64; I=torch.eye(2,dtype=ct,device=device); X=torch.tensor([[0,1],[1,0]],dtype=ct,device=device); Y=torch.tensor([[0,-1j],[1j,0]],dtype=ct,device=device); Z=torch.tensor([[1,0],[0,-1]],dtype=ct,device=device)
        self.I2=I; self.I4=torch.eye(4,dtype=ct,device=device); self.I8=torch.eye(8,dtype=ct,device=device); self.X,self.Y,self.Z=X,Y,Z; self.device=device
        self.Xs=[kron3(X,I,I),kron3(I,X,I),kron3(I,I,X)]; self.Ys=[kron3(Y,I,I),kron3(I,Y,I),kron3(I,I,Y)]; self.Zs=[kron3(Z,I,I),kron3(I,Z,I),kron3(I,I,Z)]
        self.Hx=sum(self.Xs); self.Hy=sum(self.Ys); self.pair_ops=[]
        for i,j in [(0,1),(0,2),(1,2)]: self.pair_ops.append(self.Xs[i]@self.Xs[j]+self.Ys[i]@self.Ys[j]+self.Zs[i]@self.Zs[j])
        self.H0=self.build_h0(torch.tensor(cfg.v_hz,dtype=torch.float32,device=device)[None,:],torch.tensor(cfg.j_hz,dtype=torch.float32,device=device)[None,:])[0]
    def build_h0(self,v,j):
        H=torch.zeros((len(v),8,8),dtype=torch.complex64,device=self.device)
        for i in range(3): H=H+math.pi*v[:,i,None,None].to(H.dtype)*self.Zs[i]
        for k in range(3): H=H+math.pi*j[:,k,None,None].to(H.dtype)*self.pair_ops[k]
        return H
SYS=NMRSystem(CFG,DEVICE)
def sample_haar(n,seed):
    g=torch.Generator(device=DEVICE); g.manual_seed(seed); q=torch.randn(n,4,generator=g,device=DEVICE); return q/q.norm(dim=-1,keepdim=True).clamp_min(1e-12)
def q2u(q):
    q0,qx,qy,qz=q.unbind(-1); I,X,Y,Z=SYS.I2,SYS.X,SYS.Y,SYS.Z
    return q0[:,None,None].to(I.dtype)*I-1j*(qx[:,None,None].to(I.dtype)*X+qy[:,None,None].to(I.dtype)*Y+qz[:,None,None].to(I.dtype)*Z)
def lift(U2): return torch.einsum('bij,kl->bikjl',U2,SYS.I4).reshape(-1,8,8)
class PulseCompiler(nn.Module):
    def __init__(self,cfg):
        super().__init__(); L=[]; d=4
        for _ in range(cfg.hidden_layers):
            L += [nn.Linear(d,cfg.width),nn.GELU()]
            if cfg.dropout>0:L.append(nn.Dropout(cfg.dropout))
            d=cfg.width
        self.backbone=nn.Sequential(*L); self.head=nn.Linear(d,2*cfg.n_slices); self.cfg=cfg
    def forward(self,q):
        r=self.head(self.backbone(q)).view(-1,self.cfg.n_slices,2); return self.cfg.rabi_max_hz*r/torch.sqrt(1+(r*r).sum(-1,keepdim=True))
def fidelity(T,U):
    o=(T.conj()*U).sum((-2,-1)); return o.abs().square()/64.0
def prop_nom(c):
    ux,uy=c[...,0],c[...,1]; H=SYS.H0[None,None]+math.pi*ux[...,None,None].to(torch.complex64)*SYS.Hx+math.pi*uy[...,None,None].to(torch.complex64)*SYS.Hy; Us=torch.matrix_exp((-1j*CFG.dt_s)*H); U=SYS.I8[None].expand(len(c),-1,-1).clone()
    for t in range(CFG.n_slices): U=Us[:,t]@U
    return U
def summary(x):
    x=np.asarray(x,float); return {'mean':float(x.mean()),'std':float(x.std()),'median':float(np.median(x)),'p05':float(np.quantile(x,.05)),'p01':float(np.quantile(x,.01)),'min':float(x.min()),'max':float(x.max()),'frac_ge_099':float(np.mean(x>=.99)),'frac_ge_098':float(np.mean(x>=.98)),'frac_ge_095':float(np.mean(x>=.95))}
ckpt=torch.load(BEST_PATH,map_location=DEVICE,weights_only=False); MODEL=PulseCompiler(CFG).to(DEVICE); MODEL.load_state_dict(ckpt['model'] if isinstance(ckpt,dict) and 'model' in ckpt else ckpt); MODEL.eval(); [p.requires_grad_(False) for p in MODEL.parameters()]
print('Loaded frozen nominal weights.'); assert abs(CFG.dt_s-35e-6)<1e-12; assert CFG.hidden_layers==10

In [ ]:
N_NEW_GATES=256; N_SWEEP_GATES=128; N_MAP_GATES=64; N_JOINT_SCENARIOS=96; GATE_CHUNK=8; SCENARIO_CHUNK=8; STRESS_SEED=20260918
SWEEPS={'common_B0_hz':np.linspace(-10,10,21),'rf_gain_fraction':np.linspace(-.15,.15,25),'phase_bias_deg':np.linspace(-3,3,25),'J_magnitude_offset_hz':np.linspace(-1,1,21),'clock_error_ppm':np.linspace(-5,5,21),'spin1_offset_hz':np.linspace(-6,6,21),'spin2_offset_hz':np.linspace(-6,6,21),'spin3_offset_hz':np.linspace(-6,6,21)}
PROFILES={'routine':{'common_B0_abs_hz':2.0,'independent_v_abs_hz':2.0,'independent_J_abs_hz':.25,'rf_gain_abs_fraction':.05,'phase_bias_abs_deg':1.0,'clock_abs_ppm':2.0,'timing_jitter_rms_ps':10.0},'conservative':{'common_B0_abs_hz':6.0,'independent_v_abs_hz':6.0,'independent_J_abs_hz':1.0,'rf_gain_abs_fraction':.15,'phase_bias_abs_deg':3.0,'clock_abs_ppm':5.0,'timing_jitter_rms_ps':50.0}}
def nominal_scen(S=1): return {'common_B0_hz':np.zeros(S,np.float32),'v_offset_hz':np.zeros((S,3),np.float32),'J_offset_hz':np.zeros((S,3),np.float32),'rf_gain':np.ones(S,np.float32),'phase_bias_rad':np.zeros(S,np.float32),'clock_ppm':np.zeros(S,np.float32),'timing_jitter_s':np.zeros((S,CFG.n_slices),np.float32)}
def sweep_scen(name,val):
    val=np.asarray(val,np.float32); s=nominal_scen(len(val))
    if name=='common_B0_hz':s['common_B0_hz']=val
    elif name=='rf_gain_fraction':s['rf_gain']=1+val
    elif name=='phase_bias_deg':s['phase_bias_rad']=np.deg2rad(val).astype(np.float32)
    elif name=='J_magnitude_offset_hz':s['J_offset_hz']=val[:,None]*np.sign(np.asarray(CFG.j_hz,np.float32))[None,:]
    elif name=='clock_error_ppm':s['clock_ppm']=val
    elif name.startswith('spin'): s['v_offset_hz'][:,int(name[4])-1]=val
    return s
def joint_scen(name,S,seed):
    p=PROFILES[name]; z=2*torch.quasirandom.SobolEngine(10,scramble=True,seed=seed).draw(S).numpy().astype(np.float32)-1; s=nominal_scen(S); k=0
    s['common_B0_hz']=z[:,k]*p['common_B0_abs_hz']; k+=1; s['v_offset_hz']=z[:,k:k+3]*p['independent_v_abs_hz']; k+=3; s['J_offset_hz']=z[:,k:k+3]*p['independent_J_abs_hz']; k+=3; s['rf_gain']=1+z[:,k]*p['rf_gain_abs_fraction']; k+=1; s['phase_bias_rad']=np.deg2rad(z[:,k]*p['phase_bias_abs_deg']).astype(np.float32); k+=1; s['clock_ppm']=z[:,k]*p['clock_abs_ppm']
    s['timing_jitter_s']=np.random.default_rng(seed+999).normal(0,p['timing_jitter_rms_ps']*1e-12,(S,CFG.n_slices)).astype(np.float32); return s
def prop_pert(c,targ,s):
    G,S=len(c),len(s['rf_gain']); out=np.empty((G,S),np.float32); v0=torch.tensor(CFG.v_hz,dtype=torch.float32,device=DEVICE); j0=torch.tensor(CFG.j_hz,dtype=torch.float32,device=DEVICE)
    for g0 in range(0,G,GATE_CHUNK):
      g1=min(G,g0+GATE_CHUNK); cc=c[g0:g1]; tt=targ[g0:g1]; Gc=g1-g0
      for s0 in range(0,S,SCENARIO_CHUNK):
        s1=min(S,s0+SCENARIO_CHUNK); st={k:torch.as_tensor(v[s0:s1],device=DEVICE) for k,v in s.items()}; Sc=s1-s0; H0=SYS.build_h0(v0[None]+st['common_B0_hz'][:,None]+st['v_offset_hz'],j0[None]+st['J_offset_hz']); ce=cc[:,None].expand(Gc,Sc,-1,-1); ph=st['phase_bias_rad'][None,:,None]; cp,sp=torch.cos(ph),torch.sin(ph); gain=st['rf_gain'][None,:,None]; ux=(gain*(cp*ce[...,0]-sp*ce[...,1])).reshape(Gc*Sc,CFG.n_slices); uy=(gain*(sp*ce[...,0]+cp*ce[...,1])).reshape(Gc*Sc,CFG.n_slices); H0=H0[None].expand(Gc,Sc,-1,-1).reshape(Gc*Sc,8,8); H=H0[:,None]+math.pi*ux[...,None,None].to(torch.complex64)*SYS.Hx+math.pi*uy[...,None,None].to(torch.complex64)*SYS.Hy; dt=(CFG.dt_s*(1+st['clock_ppm']*1e-6)[:,None]+st['timing_jitter_s']); dt=dt[None].expand(Gc,Sc,-1).reshape(Gc*Sc,CFG.n_slices); Us=torch.matrix_exp((-1j*dt[...,None,None])*H); U=SYS.I8[None].expand(Gc*Sc,-1,-1).clone()
        for k in range(CFG.n_slices):U=Us[:,k]@U
        T=tt[:,None].expand(Gc,Sc,-1,-1).reshape(Gc*Sc,8,8); out[g0:g1,s0:s1]=fidelity(T,U).reshape(Gc,Sc).cpu().numpy(); del H,Us,U
      torch.cuda.empty_cache()
    return out
def cvar_inf(F,a=.1):
    L=1-F; k=max(1,int(np.ceil(a*L.shape[1]))); return np.partition(L,L.shape[1]-k,axis=1)[:,L.shape[1]-k:].mean(1)
def risk_table(F,nom): return pd.DataFrame({'gate_index':np.arange(len(F)),'nominal_fidelity':nom,'mean_fidelity':F.mean(1),'median_fidelity':np.median(F,axis=1),'p05_fidelity':np.quantile(F,.05,axis=1),'worst_fidelity':F.min(1),'cvar10_infidelity':cvar_inf(F,.1),'cvar20_infidelity':cvar_inf(F,.2)})

In [ ]:
STAMP=datetime.now().strftime('%Y%m%d_%H%M%S'); OUT=REPO_ROOT/'outputs'/f'frozen_robustness_stress_{STAMP}'; RAW=OUT/'raw'; TABLES=OUT/'tables'; FIGS=OUT/'figures'; SW=RAW/'sweeps'; JO=RAW/'joint'
for d in [OUT,RAW,TABLES,FIGS,SW,JO]:d.mkdir(parents=True,exist_ok=True)
shutil.copy2(CONFIG_PATH,OUT/'nominal_config.json'); shutil.copy2(BEST_PATH,OUT/'frozen_best_checkpoint.pt')
with open(OUT/'stress_profiles.json','w') as f:json.dump(PROFILES,f,indent=2)
with open(OUT/'literature_basis_and_ranges.json','w') as f:json.dump({'RF_B1':'routine ±5%, conservative ±15%; NMR robust-pulse literature including Nimbalkar et al. JMR 2013','B0':'routine ±2 Hz common/independent, conservative ±6 Hz; few-Hz/hour high-resolution NMR scale, treated quasi-static','J':'routine ±0.25 Hz, conservative ±1 Hz; qNMR and solvent-dependence literature','timing':'routine ±2 ppm + 10 ps RMS, conservative ±5 ppm + 50 ps RMS; HDAWG/Tek AWG specs','phase':'±1° routine, ±3° conservative engineering calibration bound'},f,indent=2)
@torch.inference_mode()
def compile_gates(q):
    cc=[];tt=[];ff=[]
    for a in range(0,len(q),32):
        x=q[a:a+32]; T=lift(q2u(x)); c=MODEL(x); U=prop_nom(c); cc.append(c.cpu()); tt.append(T.cpu()); ff.append(fidelity(T,U).cpu())
    return torch.cat(cc).to(DEVICE),torch.cat(tt).to(DEVICE),torch.cat(ff).numpy()
if ARCHIVED_Q_PATH and ARCHIVED_FIDS_PATH:
    q=torch.tensor(np.load(ARCHIVED_Q_PATH),dtype=torch.float32,device=DEVICE); stored=np.load(ARCHIVED_FIDS_PATH); _,_,rec=compile_gates(q); diff=float(np.max(np.abs(stored-rec))); print('Archive reproduction max diff',diff); assert diff<2e-4
q=sample_haar(N_NEW_GATES,STRESS_SEED); controls,targets,nom=compile_gates(q); np.save(RAW/'new_haar_quaternions.npy',q.cpu().numpy()); np.save(RAW/'new_nominal_fidelities.npy',nom); nomsum=summary(nom); print('NEW nominal',json.dumps(nomsum,indent=2)); pd.DataFrame({'gate_index':np.arange(N_NEW_GATES),'q0':q[:,0].cpu(),'qx':q[:,1].cpu(),'qy':q[:,2].cpu(),'qz':q[:,3].cpu(),'nominal_fidelity':nom}).to_csv(TABLES/'new_haar_nominal_gates.csv',index=False)
frames=[]
for name,val in SWEEPS.items():
    print('SWEEP',name); F=prop_pert(controls[:N_SWEEP_GATES],targets[:N_SWEEP_GATES],sweep_scen(name,val)); np.save(SW/f'{name}.npy',F); df=pd.DataFrame([{'parameter':name,'value':float(x),**summary(F[:,i])} for i,x in enumerate(val)]); df.to_csv(TABLES/f'sweep_{name}.csv',index=False); frames.append(df)
pd.concat(frames).to_csv(TABLES/'all_sweep_summaries.csv',index=False)
joint={}; rows=[]
for i,name in enumerate(PROFILES):
    print('JOINT',name); sc=joint_scen(name,N_JOINT_SCENARIOS,STRESS_SEED+1000*(i+1)); np.savez(JO/f'{name}_scenarios.npz',**sc); F=prop_pert(controls,targets,sc); np.save(JO/f'{name}_fidelity_matrix.npy',F); joint[name]=F; rt=risk_table(F,nom); rt.to_csv(TABLES/f'{name}_gate_risk.csv',index=False); rows.append({'profile':name,**{f'scenario_{k}':v for k,v in summary(F.ravel()).items()},**{f'gate_mean_{k}':v for k,v in summary(F.mean(1)).items()},**{f'gate_worst_{k}':v for k,v in summary(F.min(1)).items()},'mean_cvar10_infidelity':float(rt.cvar10_infidelity.mean()),'mean_cvar20_infidelity':float(rt.cvar20_infidelity.mean())})
PS=pd.DataFrame(rows); PS.to_csv(TABLES/'joint_profile_summary.csv',index=False); display(PS)
B0=np.linspace(-6,6,13,dtype=np.float32); GG=np.linspace(-.15,.15,13,dtype=np.float32); pairs=[(b,g) for b in B0 for g in GG]; sc=nominal_scen(len(pairs)); sc['common_B0_hz']=np.array([x[0] for x in pairs],np.float32); sc['rf_gain']=1+np.array([x[1] for x in pairs],np.float32); FM=prop_pert(controls[:N_MAP_GATES],targets[:N_MAP_GATES],sc); MM=FM.mean(0).reshape(13,13); PM=np.quantile(FM,.05,axis=0).reshape(13,13); np.savez(RAW/'B0_RF_gain_2d_map.npz',B0_hz=B0,gain_fraction=GG,fidelity_matrix=FM,mean_map=MM,p05_map=PM)

In [ ]:
def sf(n):plt.tight_layout();plt.savefig(FIGS/f'{n}.png',dpi=600,bbox_inches='tight');plt.savefig(FIGS/f'{n}.pdf',bbox_inches='tight');plt.show();plt.close()
plt.figure(figsize=(7,4.6));plt.hist(nom,bins=35);plt.axvline(np.mean(nom),ls='--',label=f'mean={np.mean(nom):.4f}');plt.axvline(np.quantile(nom,.05),ls=':',label=f'p05={np.quantile(nom,.05):.4f}');plt.xlabel('Nominal fidelity');plt.ylabel('Unseen Haar gates');plt.legend();plt.grid(alpha=.15);sf('new_haar_nominal_histogram')
for name in SWEEPS:
    d=pd.read_csv(TABLES/f'sweep_{name}.csv');plt.figure(figsize=(7,4.6))
    for col in ['mean','median','p05','min']:plt.plot(d.value,d[col],marker='o',ms=3,label=col)
    plt.xlabel(name);plt.ylabel('Unitary fidelity');plt.ylim(0,1.005);plt.legend();plt.grid(alpha=.2);sf(f'sweep_{name}')
for name,F in joint.items():
    plt.figure(figsize=(7,4.6));plt.hist(F.ravel(),bins=45);plt.xlabel('Gate × scenario fidelity');plt.ylabel('Count');plt.title(name);plt.grid(alpha=.15);sf(f'joint_{name}_histogram')
    r=pd.read_csv(TABLES/f'{name}_gate_risk.csv');plt.figure(figsize=(6.2,5.2));plt.scatter(r.nominal_fidelity,r.mean_fidelity,s=14,alpha=.7);lo=min(r.nominal_fidelity.min(),r.mean_fidelity.min());plt.plot([lo,1],[lo,1],'--',lw=1);plt.xlabel('Nominal fidelity');plt.ylabel('Mean perturbed fidelity');plt.grid(alpha=.2);sf(f'joint_{name}_nominal_vs_mean')
    plt.figure(figsize=(7,4.6));plt.hist(r.worst_fidelity,bins=35,alpha=.8,label='worst scenario');plt.hist(1-r.cvar10_infidelity,bins=35,alpha=.55,label='1 - CVaR10 infidelity');plt.xlabel('Gate-level robustness score');plt.ylabel('Gate count');plt.legend();plt.grid(alpha=.15);sf(f'joint_{name}_tail_metrics')
for arr,label in [(MM,'mean'),(PM,'p05')]:
    plt.figure(figsize=(7.2,5.4));im=plt.imshow(arr,origin='lower',aspect='auto',extent=[GG[0]*100,GG[-1]*100,B0[0],B0[-1]],vmin=max(0,float(arr.min())),vmax=1);plt.colorbar(im,label=f'{label} fidelity');plt.xlabel('RF gain error (%)');plt.ylabel('Common B0 / carrier offset (Hz)');plt.title(f'{label.upper()} fidelity: B0 × RF gain');sf(f'B0_RF_gain_{label}_heatmap')

In [ ]:
report={'checkpoint_step':ckpt.get('step'),'checkpoint_stats':ckpt.get('val_stats'),'checkpoint_sha256':sha256_file(BEST_PATH),'physics':{'v_hz':list(CFG.v_hz),'J_hz':list(CFG.j_hz),'dt_s':CFG.dt_s,'n_slices':CFG.n_slices,'total_pulse_s':CFG.dt_s*CFG.n_slices,'interaction':'full XX+YY+ZZ'},'new_nominal':nomsum,'profiles':PROFILES}
with open(OUT/'stress_test_report.json','w') as f:json.dump(report,f,indent=2,default=str)
with open(OUT/'README_RESULTS.txt','w') as f:
    f.write('FROZEN FULL-HAAR NMR ROBUSTNESS STRESS TEST\nNo retraining was performed.\n\nNEW UNTOUCHED HAAR NOMINAL\n'+json.dumps(nomsum,indent=2)+'\n\nJOINT SUMMARY\n'+PS.to_string(index=False))
env={'python':sys.version,'platform':platform.platform(),'torch':torch.__version__,'numpy':np.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}
with open(OUT/'environment.json','w') as f:json.dump(env,f,indent=2)
with open(OUT/'MANIFEST.txt','w') as f:f.write('\n'.join(str(p.relative_to(OUT)) for p in sorted(OUT.rglob('*')) if p.is_file()))
with open(OUT/'SHA256SUMS.txt','w') as f:f.write('\n'.join(f'{sha256_file(p)}  {p.relative_to(OUT)}' for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='SHA256SUMS.txt'))
ZIP=shutil.make_archive(str(OUT),'zip',root_dir=OUT.parent,base_dir=OUT.name);print('ZIP',ZIP,f'{os.path.getsize(ZIP)/1024**2:.2f} MB');display(PS);files.download(ZIP)